# LEGB, Global, Nonlocal, Enclosing e Call Stack de funções em Python

A regra LEGB e como o Python a usa para resolver nomes

Modificar o comportamento do Python com `global` e `nonlocal`

varnames, freevars e cellvars

## Regra LEGB

Como o Python resolve nomes:

O Python segue uma ordem específica e unidirecional para busca por nomes. A ordem sempre vai do escopo mais interno para o mais externo:

Local --> Enclosing --> Global --> Built-in --> Se não entrar (NameError)

De nenhum escopo externo é possível usar algo de escopo interno.

In [3]:
_print = print

nome_global = "nome_global"

def func_global() -> None:
    nome_enclosing = "nome_enclosing" # Enclosing (Local)

    def print(*args):
        _print('Meu print:', *args)

    def func_interna() -> None:
        nome_local = "nome_local" # Local

        print(
            "Local:",
            nome_local,
            nome_enclosing,
            "funcao_interna",
            nome_global,
            "+built-ins",
        )
    func_interna()
    # print(
    #         "Enclosing:",
    #         nome_enclosing,
    #         "funcao_interna",
    #         "funcao_global",
    #         nome_global,
    #         "+built-ins",
    #     )

func_global()
# print("GLOBAL:", nome_global, "func_global", "+built-ins")

Meu print: Local: nome_local nome_enclosing funcao_interna nome_global +built-ins


Uso do `global` e `nonlocal` para mudar o comportamento

Quando você define um nome em determinado escopo, o Python assume que aquele nome é único naquele escopo. Por isso, é impossível modificar o valor de um nome do escopo externo sem informar isso ao interpretador.

- `global` -> para modificar nomes do escopo global dentro de qualquer escopo local, precisamos usar a palavra chave `global`

- `nonlocal` -> Para modificar os nomes do escopo `enclosing` dentro de qualquer escopo local, precisamos usar a palvra chave `nonlocal`

In [6]:
nome_global = "nome_global"

def func_global() -> None:
    nome_enclosing = "nome_enclosing" # Enclosing (Local)

    def func_interna() -> None:
        print(nome_enclosing)
        nome_enclosing = "novo_nome_enclosing"
        print(nome_enclosing)

    func_interna()

func_global()

UnboundLocalError: cannot access local variable 'nome_enclosing' where it is not associated with a value

In [21]:
nome_global = "nome_global"
print(nome_global, id(nome_global))

def func_global() -> None:
    nome_enclosing = "Foda" # Enclosing (Local)

    def func_interna() -> None:
        global nome_global
        nonlocal nome_enclosing
        print(nome_enclosing, id(nome_enclosing))

        nome_enclosing = "Legal"
        print(nome_enclosing, id(nome_enclosing))

        nome_global = "novo_nome_global"
        print(nome_global, id(nome_global))

    func_interna()
    print(nome_enclosing, id(nome_enclosing))

func_global()

nome_global 1716936160688
Foda 1716936281088
Legal 1716936394288
novo_nome_global 1716944344688
Legal 1716936394288


varnames, freevars e cellvars

Em alguns momentos você pode ver um comportamento estranho ao solicitar o namespace local de uma função. Ao LER uma variável do enclosing, ela pode aparecer como parte do namespace local (da a impressão que ela foi definida internamente na função). O que é isso?

Detalhe: isso pode mudar dependendo do interpretador que você usar.

- Freevars são as variáveis da função externa que estão sendo usadas dentro da função interna. A gente detecta isso pela função interna, porque ela é quem depende desses nomes. Eles entram em co_freevars.

- Cellvars são variáveis declaradas na função atual (externa) que precisam ser capturadas porque são usadas por funções internas. A gente detecta isso pela função externa, porque ela é quem fornece essas variáveis pro closure. Eles aparecem em co_cellvars.

- Varnames são as variáveis locais de verdade, exclusivas da função. Elas estão em co_varnames e não fazem parte de nenhuma closure, só existem ali dentro mesmo.

In [23]:
nome_global = "nome_global"

def func_global() -> None:
    nome_enclosing = nome_global

    def func_interna() -> None:
        nome_local = nome_enclosing

        print("dir/locals func_interna:", ", ".join(dir()))

    func_interna()
    print("dir/locals func_global:", ", ".join(dir()))

func_global()
print("dir/global:", ", ".join(dir()))


dir/locals func_interna: nome_enclosing, nome_local
dir/locals func_global: func_interna, nome_enclosing
dir/global: In, Out, _, __, ___, __builtin__, __builtins__, __doc__, __loader__, __name__, __package__, __spec__, __vsc_ipynb_file__, _dh, _i, _i1, _i10, _i11, _i12, _i13, _i14, _i15, _i16, _i17, _i18, _i19, _i2, _i20, _i21, _i22, _i23, _i3, _i4, _i5, _i6, _i7, _i8, _i9, _ih, _ii, _iii, _oh, _print, exit, func_global, get_ipython, nome_global, open, quit
